In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

In [2]:
data = pd.read_csv('student_performance_prediction_dataset-2.csv')

In [3]:
data.head()

,student_id,age,gender,study_hours,attendance,sleep_hours,previous_grade,assignments_completed,practice_tests_taken,group_study_hours,...,social_media_hours,family_income,parent_education,internet_access,device_type,school_type,extracurriculars,final_grade,grade_category,pass_fail
0,1,21,Male,1.645404,79.154521,8.230886,96.053840,7.719620,1.871170,1.447894,...,4.018412,Medium,Master,Yes,Mobile,Private,Coding Club,59.248749,D,Pass
1,2,18,Male,4.462126,72.526685,6.139219,53.024821,6.754758,5.630071,1.891288,...,3.268642,Medium,Master,Yes,Laptop,Public,NaN,58.595595,D,Pass
2,3,19,Female,6.220212,98.531716,6.946313,78.775422,10.000000,7.862877,1.774356,...,2.327293,Low,High School,Yes,Tablet,Private,Music,85.855289,A,Pass
3,4,21,Female,1.826644,97.731245,8.297048,76.122618,7.440486,2.316252,1.204271,...,1.163367,Medium,Bachelor,Yes,Laptop,Public,Debate,42.117503,F,Fail
4,5,17,Male,3.789322,78.589107,6.777171,81.305681,9.962609,5.335697,1.399230,...,0.411183,High,High School,Yes,Laptop,Private,Debate,62.870474,C,Pass


In [4]:
data.isnull().sum()


,0
student_id,0
age,0
gender,0
study_hours,0
attendance,1
sleep_hours,1
previous_grade,1
assignments_completed,1
practice_tests_taken,1
group_study_hours,1


In [5]:
X = data.drop(['student_id','final_grade'],axis=1)
y = data['final_grade']


In [6]:
X['device_type'] = X['device_type'].fillna(X['device_type'].mode()[0])
X['extracurriculars'] = X['extracurriculars'].fillna(X['extracurriculars'].mode()[0])
X['grade_category'] = X['grade_category'].fillna(X['grade_category'].mode()[0])

In [7]:
X.isnull().sum()

,0
age,0
gender,0
study_hours,0
attendance,1
sleep_hours,1
previous_grade,1
assignments_completed,1
practice_tests_taken,1
group_study_hours,1
notes_quality_score,1


In [8]:
data.duplicated().sum()

np.int64(0)

In [9]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278723 entries, 0 to 278722
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   age                    278723 non-null  int64  
 1   gender                 278723 non-null  object 
 2   study_hours            278723 non-null  float64
 3   attendance             278722 non-null  float64
 4   sleep_hours            278722 non-null  float64
 5   previous_grade         278722 non-null  float64
 6   assignments_completed  278722 non-null  float64
 7   practice_tests_taken   278722 non-null  float64
 8   group_study_hours      278722 non-null  float64
 9   notes_quality_score    278722 non-null  float64
 10  time_management_score  278722 non-null  float64
 11  motivation_level       278722 non-null  float64
 12  mental_health_score    278722 non-null  float64
 13  screen_time            278722 non-null  float64
 14  social_media_hours     278722 non-nu

In [10]:
X.gender.unique()

array(['Male', 'Female', 'Other'], dtype=object)

In [11]:
from sklearn.preprocessing import LabelEncoder
lbl_enc = LabelEncoder()

In [12]:
X['gender'] = lbl_enc.fit_transform(X[['gender']])

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [13]:
X.gender.unique()

array([1, 0, 2])

In [14]:
X.family_income.unique()


array(['Medium', 'Low', 'High', nan], dtype=object)

In [15]:
from sklearn.preprocessing import OrdinalEncoder
ord_enc = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])

In [20]:
from sklearn.preprocessing import OrdinalEncoder

X['family_income'] = X['family_income'].fillna('Missing')

ord_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X['family_income'] = ord_enc.fit_transform(X[['family_income']])


In [21]:
X.family_income.unique()

array([2., 1., 0., 3.])

In [22]:
X.parent_education.unique()

array(['Master', 'High School', 'Bachelor', 'PhD', nan], dtype=object)

In [23]:
ord_enc = OrdinalEncoder(categories=[['High School','Bachelor','Master','PhD']])

In [26]:
from sklearn.preprocessing import OrdinalEncoder

X['parent_education'] = X['parent_education'].astype(str)

ord_enc_parent = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X['parent_education'] = ord_enc_parent.fit_transform(
    X[['parent_education']]
)


In [27]:
X.parent_education.unique()


array([2., 1., 0., 4., 3.])

In [28]:
X.internet_access.unique()


array(['Yes', 'No', nan], dtype=object)

In [29]:
X.internet_access = X.internet_access.apply(lambda x:1 if x == 'Yes' else 0)


In [30]:
X.internet_access.unique()

array([1, 0])

In [31]:
X.device_type.unique()

array(['Mobile', 'Laptop', 'Tablet'], dtype=object)

In [32]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

In [33]:
encoded = ohe.fit_transform(X[['device_type']])

In [34]:
encoded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(['device_type']),
                          index=X.index)

In [35]:
X = pd.concat([X.drop('device_type',axis=1),encoded_df],axis=1)

In [36]:
encoded_df.head()

,device_type_Laptop,device_type_Mobile,device_type_Tablet
0,0.0,1.0,0.0
1,1.0,0.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0


In [37]:
X.school_type.unique()

array(['Private', 'Public', nan], dtype=object)

In [38]:
X['school_type'] = X.school_type.apply(lambda x:1 if x=='Private' else 0)

In [39]:
X['school_type'].unique()

array([1, 0])

In [40]:
X.extracurriculars.unique()

array(['Coding Club', 'Music', 'Debate', 'Sports', 'Arts'], dtype=object)

In [41]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
encoded_extcurr = ohe.fit_transform(X[['extracurriculars']])


In [42]:
enc_extcurr_df = pd.DataFrame(encoded_extcurr, columns= ohe.get_feature_names_out(['extracurriculars']),
                              index=X.index)


In [43]:
enc_extcurr_df.head()


,extracurriculars_Arts,extracurriculars_Coding Club,extracurriculars_Debate,extracurriculars_Music,extracurriculars_Sports
0,0.0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0


In [44]:
X = pd.concat([X.drop('extracurriculars',axis=1),enc_extcurr_df],axis=1)


In [45]:
X.grade_category.unique()


array(['D', 'A', 'F', 'C', 'B', 'A+'], dtype=object)

In [46]:
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[['F', 'D', 'C', 'B', 'A', 'A+']])


In [47]:
X['grade_category'] = oe.fit_transform(X[['grade_category']])

In [48]:
X['grade_category'].unique()

array([1., 4., 0., 2., 3., 5.])

In [49]:
X.pass_fail.unique()

array(['Pass', 'Fail', nan], dtype=object)

In [50]:
X['pass_fail'] = X.pass_fail.apply(lambda x:1 if x=='Pass' else 0)

In [51]:
X.pass_fail.unique()

array([1, 0])

In [52]:
X.head()

,age,gender,study_hours,attendance,sleep_hours,previous_grade,assignments_completed,practice_tests_taken,group_study_hours,notes_quality_score,...,grade_category,pass_fail,device_type_Laptop,device_type_Mobile,device_type_Tablet,extracurriculars_Arts,extracurriculars_Coding Club,extracurriculars_Debate,extracurriculars_Music,extracurriculars_Sports
0,21,1,1.645404,79.154521,8.230886,96.053840,7.719620,1.871170,1.447894,6.811654,...,1.0,1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,18,1,4.462126,72.526685,6.139219,53.024821,6.754758,5.630071,1.891288,10.000000,...,1.0,1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,19,0,6.220212,98.531716,6.946313,78.775422,10.000000,7.862877,1.774356,6.484908,...,4.0,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,21,0,1.826644,97.731245,8.297048,76.122618,7.440486,2.316252,1.204271,6.057978,...,0.0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,17,1,3.789322,78.589107,6.777171,81.305681,9.962609,5.335697,1.399230,9.013332,...,2.0,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [53]:
y.head()

,final_grade
0,59.248749
1,58.595595
2,85.855289
3,42.117503
4,62.870474


In [54]:

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size= .2, random_state=42)


In [55]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((222978, 29), (55745, 29), (222978,), (55745,))

In [56]:
from sklearn.preprocessing import StandardScaler
std_scaler = StandardScaler()

In [57]:
X_train_std_scl = std_scaler.fit_transform(X_train)
X_test_std_scl = std_scaler.transform(X_test)


In [58]:
X_train_std_scl.shape[1]

29

In [59]:
model = keras.Sequential([
    layers.Input(shape=(X_train_std_scl.shape[1],)),
    layers.Dense(64,activation='relu'),
    layers.Dense(32,activation='relu'),
    layers.Dense(16,activation='relu'),
    layers.Dense(1,activation='softmax'),
])

In [60]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,545 (17.75 KB)

 Trainable params: 4,545 (17.75 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
model.compile(optimizer='adam', loss='mse' , metrics=['mae'])

In [62]:
model.fit(X_train_std_scl, y_train, epochs=20, batch_size=32)

Epoch 1/20


/usr/local/lib/python3.13/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


6969/6969 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - loss: nan - mae: nan
Epoch 2/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - loss: nan - mae: nan
Epoch 3/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: nan - mae: nan
Epoch 4/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: nan - mae: nan
Epoch 5/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - loss: nan - mae: nan
Epoch 6/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 41s 4ms/step - loss: nan - mae: nan
Epoch 7/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: nan - mae: nan
Epoch 8/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: nan - mae: nan
Epoch 9/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: nan - mae: nan
Epoch 10/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - loss: nan - mae: nan
Epoch 11/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: nan - mae: nan
Epoch 12/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: nan - mae: nan
Epoch 13/20
6969/6969 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: nan - ma

In [63]:
def build_ann(activation='relu', optimizer='adam', input_shape=X_train_std_scl.shape[1]):
    model = Sequential([
        layers.Dense(64, activation=activation, input_shape=(input_shape,)),
        layers.Dense(32, activation=activation),
        layers.Dense(16, activation=activation),
        layers.Dense(1)  # Continuous linear output for regression
    ])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

In [64]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [65]:
experiments = [
    {"name": "Model 1", "activation": "relu", "optimizer": "adam", "epochs": 20, "batch_size": 64},
    {"name": "Model 2", "activation": "tanh", "optimizer": "sgd", "epochs": 50, "batch_size": 64},
    {"name": "Model 3", "activation": "sigmoid", "optimizer": "rmsprop", "epochs": 100, "batch_size": 64},
    {"name": "Model 4 (Optimal)", "activation": "relu", "optimizer": "adam", "epochs": 50, "batch_size": 32}
]

results = []

In [ ]:
for exp in experiments:
    model = build_ann(activation=exp['activation'], optimizer=exp['optimizer'])
    history = model.fit(
        X_train_std_scl, y_train,
        validation_split=0.2,
        epochs=exp['epochs'],
        batch_size=exp['batch_size'],
        verbose=0
    )

    preds = model.predict(X_test_std_scl, verbose=0).flatten()
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        "Model": exp['name'],
        "Activation": exp['activation'].upper(),
        "Optimizer": exp['optimizer'].upper(),
        "Epochs": exp['epochs'],
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2 Score": round(r2, 4)
    })

In [71]:
results


[{'Model': 'Model 1',
  'Activation': 'RELU',
  'Optimizer': 'ADAM',
  'Epochs': 20,
  'MAE': 2.5709,
  'RMSE': np.float64(3.2767),
  'R2 Score': 0.9281},
 {'Model': 'Model 1',
  'Activation': 'RELU',
  'Optimizer': 'ADAM',
  'Epochs': 20,
  'MAE': 2.6341,
  'RMSE': np.float64(3.3364),
  'R2 Score': 0.9255},
 {'Model': 'Model 2',
  'Activation': 'TANH',
  'Optimizer': 'SGD',
  'Epochs': 50,
  'MAE': 2.5934,
  'RMSE': np.float64(3.3329),
  'R2 Score': 0.9256}]